# 보간법 baseline — Nearest / Bilinear / Bicubic / Lanczos

선명한 위성사진을 ×2, ×3, ×4 로 줄였다가 4가지 보간법으로 되돌려 PSNR·SSIM 을 잰다.
딥러닝 SR 과 비교할 기준선을 만든다. GPU 불필요.

## 1. 데이터

In [ ]:
import json, os, urllib.request
import numpy as np, imageio.v2 as imageio, matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
REP  = {'training': 'AOI_Barcelona_10_y0128_x0128', 'validation': 'AOI_Paris_1_6_y0064_x0192'}
TEST = 'incheon_600.png'

def fetch(url, path):
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path

def pair(split, stem):
    """(입력 LR, 정답 HR)"""
    hr = imageio.imread(fetch(f'{BASE}/dataset/{split}/HR/{stem}.png', f'{split}/{stem}.png'))
    lr = imageio.imread(fetch(f'{BASE}/dataset/{split}/LR_bicubic/X3/{stem}x3.png', f'{split}/{stem}_lr.png'))
    return lr, hr

def show(items, title=''):
    """items: [(이름, 입력, 정답 또는 None)]"""
    fig, ax = plt.subplots(2, len(items), figsize=(3.3 * len(items), 7.0), squeeze=False)
    for c, (name, lo, hi) in enumerate(items):
        ax[0, c].imshow(lo); ax[0, c].set_title(f'{name}\ninput {lo.shape[0]}px', fontsize=9)
        if hi is None:
            ax[1, c].text(.5, .5, 'no target', ha='center', va='center', fontsize=11, color='#888')
            ax[1, c].set_facecolor('#f2f2f2')
        else:
            ax[1, c].imshow(hi); ax[1, c].set_title(f'target {hi.shape[0]}px', fontsize=9)
        for r in (0, 1): ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
    if title: fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

val_lr, val_hr = pair('validation', REP['validation'])
test_lr = imageio.imread(fetch(f'{BASE}/dataset/test/{TEST}', 'test.png'))

show([('validation (Paris)', val_lr, val_hr),
      ('test (Incheon)', test_lr, None)])

## 2. 보간 적용

In [ ]:
import cv2

KERNELS = {'Nearest': cv2.INTER_NEAREST, 'Bilinear': cv2.INTER_LINEAR,
           'Bicubic': cv2.INTER_CUBIC,  'Lanczos': cv2.INTER_LANCZOS4}
SCALES = [2, 3, 4]

def down(hr, s):
    H, W = hr.shape[:2]
    return cv2.resize(hr, (W // s, H // s), interpolation=cv2.INTER_AREA)

def up(lr, s, k):
    H, W = lr.shape[:2]
    return cv2.resize(lr, (W * s, H * s), interpolation=KERNELS[k])

for s in SCALES:
    lo = down(val_hr, s)
    print(f'x{s}:  HR {val_hr.shape[1]}px  ->  LR {lo.shape[1]}px  ->  복원 {up(lo, s, "Bicubic").shape[1]}px')

## 3. 정량 평가

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

SHAVE = 4
def score(pred, gt):
    a, b = pred[SHAVE:-SHAVE, SHAVE:-SHAVE], gt[SHAVE:-SHAVE, SHAVE:-SHAVE]
    return psnr(b, a, data_range=255), ssim(b, a, data_range=255, channel_axis=2)

VAL_API = 'https://api.github.com/repos/BWMIN-Hub/SR_practice/contents/dataset/validation/HR'
with urllib.request.urlopen(VAL_API) as r:
    val_names = sorted(x['name'][:-4] for x in json.load(r))
HR = {stem: pair('validation', stem)[1] for stem in val_names}

res = {}
for s in SCALES:
    for k in KERNELS:
        v = [score(up(down(hr, s), s, k), hr) for hr in HR.values()]
        res[(s, k)] = (float(np.mean([x[0] for x in v])), float(np.mean([x[1] for x in v])))

print(f'{"":8s}' + ''.join(f'{k:>20s}' for k in KERNELS))
print(f'{"":8s}' + ''.join(f'{"PSNR":>10s}{"SSIM":>10s}' for _ in KERNELS))
for s in SCALES:
    print(f'x{s:<7d}' + ''.join(f'{res[(s,k)][0]:10.2f}{res[(s,k)][1]:10.4f}' for k in KERNELS))

In [ ]:
x = np.arange(len(SCALES)); w = 0.2
colors = ['#c96a5b', '#d9a441', '#2f6f9f', '#4f9d69']
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for j, (name, unit) in enumerate([('PSNR', ' (dB)'), ('SSIM', '')]):
    for i, k in enumerate(KERNELS):
        ax[j].bar(x + (i - 1.5) * w, [res[(s, k)][j] for s in SCALES], w, label=k, color=colors[i])
    ax[j].set_xticks(x); ax[j].set_xticklabels([f'x{s}' for s in SCALES])
    ax[j].set_title(f'{name}{unit} by scale and kernel')
    lo = min(res[(s, k)][j] for s in SCALES for k in KERNELS)
    hi = max(res[(s, k)][j] for s in SCALES for k in KERNELS)
    ax[j].set_ylim(lo - (hi - lo) * .15, hi + (hi - lo) * .15)
    ax[j].grid(axis='y', alpha=.3); ax[j].legend(fontsize=8)
plt.tight_layout(); plt.show()

with open('baseline_interpolation.json', 'w') as f:
    json.dump({f'x{s}': {k: {'psnr': round(res[(s, k)][0], 3), 'ssim': round(res[(s, k)][1], 4)}
                         for k in KERNELS} for s in SCALES}, f, indent=1)
print('baseline_interpolation.json 저장')

## 4. 결과

In [ ]:
def zoom(panels, size=110, title=''):
    """가장 복잡한 구역을 찾아 확대 비교. panels: [(이름, 이미지)]"""
    import cv2
    ref = panels[-1][1]
    e = cv2.Canny(cv2.cvtColor(ref, cv2.COLOR_RGB2GRAY), 50, 150)
    best, bs = (0, 0), -1
    for y in range(0, ref.shape[0] - size, size // 2):
        for x in range(0, ref.shape[1] - size, size // 2):
            v = e[y:y+size, x:x+size].mean()
            if v > bs: best, bs = (y, x), v
    y, x = best
    fig, ax = plt.subplots(1, len(panels), figsize=(2.9 * len(panels), 3.2))
    for a, (n, im) in zip(np.atleast_1d(ax), panels):
        a.imshow(im[y:y+size, x:x+size], interpolation='nearest')
        a.set_title(n, fontsize=9); a.set_xticks([]); a.set_yticks([])
    if title: fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

S = 4
lo = down(val_hr, S)
zoom([(k, up(lo, S, k)) for k in KERNELS] + [('Target HR', val_hr)],
     title=f'validation (Paris), x{S}')

# 링잉: 보간에 쓰인 LR 화소들의 범위를 벗어난 출력 비율
src = up(lo, S, 'Nearest')
win = np.ones((2 * S + 1, 2 * S + 1), np.uint8)
b_lo, b_hi = cv2.erode(src, win).astype(np.int16), cv2.dilate(src, win).astype(np.int16)
print(f'{"kernel":10s}{"overshoot":>12s}{"max":>8s}')
for k in KERNELS:
    u = up(lo, S, k).astype(np.int16)
    print(f'{k:10s}{float(((u > b_hi + 2) | (u < b_lo - 2)).mean() * 100):11.2f}%'
          f'{float(np.maximum(0, np.maximum(u - b_hi, b_lo - u)).max()):7.0f} DN')

In [ ]:
t_up = {k: up(test_lr, 3, k) for k in ['Nearest', 'Bicubic', 'Lanczos']}
zoom([(k, v) for k, v in t_up.items()], title='test (Incheon), x3 — no target')